# Weather Temperature Prediction

**Tabular Regression Project** — Predict surface temperature from atmospheric and geographic features.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Target: `temperature_c`

## 2 · Project Overview

We predict **surface air temperature** (°C) from atmospheric measurements (pressure, humidity, dew point, wind, cloud cover, UV index) and geographic/temporal features (latitude, elevation, month, hour). Temperature prediction is fundamental in meteorology and climate science.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Build a baseline Linear Regression model.
3. Benchmark many regressors with LazyPredict.
4. Run FLAML AutoML for automated model selection.
5. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
6. Evaluate with RMSE, MAE, R² and residual analysis.
7. Compare all models and select the top 3.

## 4 · Problem Statement

Given atmospheric readings — humidity, pressure, wind speed, cloud cover, dew point, UV index — plus geographic context (latitude, elevation) and time (month, hour), predict the **surface air temperature in °C**.

## 5 · Why This Project Matters

- **Weather forecasting** is one of the most impactful prediction problems in the world.
- Temperature affects energy demand, agriculture, transportation, and public health.
- Understanding temperature drivers builds intuition for climate science.
- ML approaches complement physics-based numerical weather prediction (NWP).
- Feature engineering (cyclical encoding, dew-point relationships) teaches transferable skills.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,500 |
| **Features** | 10 (humidity, pressure, wind, cloud cover, dew point, UV, month, hour, latitude, elevation) |
| **Target** | `temperature_c` (continuous, °C) |
| **Categorical** | None (all numeric) |
| **Missing** | None |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Free for learning and experimentation.
- **Limitations**: Simulated data — real-world relationships may be more complex.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "temperature_c"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

We generate a synthetic dataset so the notebook is fully self-contained.

In [ ]:
np.random.seed(42)
n = 8500
humidity = np.random.uniform(10, 100, n)
pressure = np.random.uniform(980, 1040, n)
wind_speed = np.random.uniform(0, 80, n)
cloud_cover = np.random.uniform(0, 100, n)
dew_point = np.random.uniform(-15, 28, n)
uv_index = np.random.uniform(0, 12, n)
month = np.random.randint(1, 13, n)
hour = np.random.randint(0, 24, n)
latitude = np.random.uniform(10, 60, n)
elevation = np.random.uniform(0, 3000, n)

# Temperature model: dew_point is strongly related, latitude and elevation provide cooling
seasonal = 10 * np.sin(2 * np.pi * (month - 4) / 12)  # peaks in July
diurnal = 5 * np.sin(2 * np.pi * (hour - 6) / 24)     # peaks at 2 PM
temp = (dew_point * 0.7 + 15
        - latitude * 0.3 - elevation * 0.005
        + uv_index * 0.8 - wind_speed * 0.05
        - (pressure - 1013) * 0.2 - cloud_cover * 0.03
        + seasonal + diurnal
        + np.random.normal(0, 2.5, n))

df = pd.DataFrame({
    "humidity_pct": humidity, "pressure_hpa": pressure, "wind_speed_kmh": wind_speed,
    "cloud_cover_pct": cloud_cover, "dew_point_c": dew_point, "uv_index": uv_index,
    "month": month, "hour": hour, "latitude": latitude, "elevation_m": elevation,
    "temperature_c": temp
})
print(f"Generated dataset: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
df.describe()

In [ ]:
num_cols = df.columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(["dew_point_c", "latitude", "elevation_m",
                          "uv_index", "pressure_hpa", "cloud_cover_pct"]):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[col], df["temperature_c"], alpha=0.2, s=8)
    ax.set_xlabel(col); ax.set_ylabel("temperature_c"); ax.set_title(f"Temp vs {col}")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.groupby("month")["temperature_c"].mean().plot(ax=axes[0], marker="o", color="steelblue")
axes[0].set_title("Mean Temperature by Month"); axes[0].set_ylabel("°C")
df.groupby("hour")["temperature_c"].mean().plot(ax=axes[1], marker="s", color="darkorange")
axes[1].set_title("Mean Temperature by Hour"); axes[1].set_ylabel("°C")
plt.tight_layout(); plt.show()

## 14 · Target Analysis

Examine the distribution of `temperature_c`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df[TARGET], bins=40, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}"); axes[0].set_xlabel(f"{TARGET} (°C)")
axes[1].boxplot(df[TARGET], vert=True); axes[1].set_title(f"Box Plot of {TARGET}")
plt.tight_layout(); plt.show()
print(f"Mean: {df[TARGET].mean():.1f}°C, Median: {df[TARGET].median():.1f}°C")
print(f"Std: {df[TARGET].std():.1f}°C, Range: {df[TARGET].min():.1f} to {df[TARGET].max():.1f}°C")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split with a fixed random seed for reproducibility.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X.columns = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing values**: None in this synthetic dataset.
- **Categorical**: Encoded via OrdinalEncoder.
- **Scaling**: Not required for tree-based models.
- **Outliers**: Handled during generation.

## 17 · Feature Engineering

In [ ]:
X_train = X_train.copy(); X_test = X_test.copy()

# Cyclical encoding for month and hour
for df_part in [X_train, X_test]:
    df_part["month_sin"] = np.sin(2 * np.pi * df_part["month"] / 12)
    df_part["month_cos"] = np.cos(2 * np.pi * df_part["month"] / 12)
    df_part["hour_sin"] = np.sin(2 * np.pi * df_part["hour"] / 24)
    df_part["hour_cos"] = np.cos(2 * np.pi * df_part["hour"] / 24)

# Dew point depression (temperature proxy)
X_train["dew_depression_proxy"] = X_train["humidity_pct"] / 100 * X_train["dew_point_c"]
X_test["dew_depression_proxy"] = X_test["humidity_pct"] / 100 * X_test["dew_point_c"]

print(f"Features ({X_train.shape[1]}): {list(X_train.columns)}")

## 18 · Baseline Model

Linear Regression provides a simple, interpretable baseline.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Quickly rank dozens of regressors.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 10 Regressors:")
print(lazy_models.head(10).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60 s budget).

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                     estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                     verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    rmse_flaml = root_mean_squared_error(y_test, y_pred_flaml)
    r2_flaml = r2_score(y_test, y_pred_flaml)
    print(f"FLAML best model: {flaml_automl.best_estimator}")
    print(f"RMSE: {rmse_flaml:,.2f}")
    print(f"R2  : {r2_flaml:.4f}")
except Exception as e:
    print(f"FLAML error: {e}")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern gradient-boosting stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# --- CatBoost ---
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    except Exception:
        cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                               verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# --- LightGBM ---
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cuda", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 device="cpu", tree_method="hist", verbosity=0,
                                 n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=50, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["LinearRegression"] = y_pred_bl
results["FLAML"] = y_pred_flaml


## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)
    axes[i].scatter(y_test, yp, alpha=0.6, s=20, edgecolors="black", linewidths=0.3)
    mn, mx = y_test.min(), y_test.max()
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_pred_vs_actual.png"), dpi=120)
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=25, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

axes[2].scatter(best_pred, np.abs(residuals), alpha=0.5, s=15, edgecolors="black", linewidths=0.3)
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "error_analysis.png"), dpi=120)
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Dew point** is the strongest individual predictor (r ~ 0.70 with temperature).
- **Latitude** and **elevation** provide the geographic baseline — higher latitude and altitude = cooler.
- **UV index** correlates with temperature through solar radiation.
- **Month** captures seasonal cycles; **hour** captures diurnal variation.
- **Pressure** has a moderate negative relationship — low pressure often brings warmer, unstable air.

**Business takeaway:** Dew point is the single most informative atmospheric variable for temperature estimation. Geographic features (latitude, elevation) set the long-term climate baseline.

## 26 · Limitations

1. Synthetic data — real temperature depends on synoptic weather patterns, fronts, and ocean currents.
2. No spatial coherence — neighboring stations should agree.
3. No lag features — yesterday's temperature predicts today's.
4. Dew point and temperature are physically linked — near-circular in some conditions.
5. No radiation or soil temperature data.

## 27 · How to Improve This Project

1. Use real weather station data (NOAA ISD, ERA5 reanalysis).
2. Add spatial features (longitude, distance to coast, land use).
3. Include lag temperature as the strongest predictor.
4. Use physics-informed ML approaches (embedding conservation laws).
5. Ensemble with NWP output for hybrid forecasting.

## 28 · Production Considerations

- Integrate with real-time sensor networks.
- Provide uncertainty estimates (prediction intervals).
- Monitor sensor drift and data quality.
- Retrain seasonally to adapt to climate trends.

## 29 · Common Mistakes

1. Using dew point without understanding its physical relationship to temperature.
2. Treating month (1-12) and hour (0-23) as linear features.
3. Not normalizing elevation across different regions.
4. Ignoring the diurnal temperature range (day vs night).
5. Evaluating on random split instead of forward-in-time split for real forecasting.

## 30 · Mini Challenge / Exercises

1. Remove `dew_point_c` — how much does R² drop? Is the model still useful?
2. Use only geographic features (latitude, elevation, month) — what R² do you get?
3. Add `wind_chill = temperature - 0.4 * wind_speed` as a target instead.
4. Try polynomial features on pressure and humidity — any improvement?
5. Compare performance using month as ordinal vs cyclical encoding.

## 31 · Final Summary / Key Takeaways

1. **Dew point** is the strongest predictor, physically linked to temperature.
2. **Latitude and elevation** set the geographic climate baseline.
3. **Cyclical encoding** of month and hour captures seasonal/diurnal patterns.
4. **Gradient boosting** handles the non-linear atmospheric relationships well.
5. **Linear Regression** is surprisingly strong due to the near-linear physics.
6. **Real weather forecasting** would benefit most from lag features and spatial data.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))